# ComfyUI + Krea 2 Turbo INT8 — KaggleUsa Cloudflare Tunnel (no necesita token ni cuenta).

In [ ]:
import os, shutil, subprocess, time, requests, threading

BASE = os.getcwd()
COMFY = os.path.join(BASE, "ComfyUI")
if os.path.exists(COMFY):
    shutil.rmtree(COMFY)

!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
os.chdir(COMFY)

!pip install -q -r requirements.txt
!pip install -q pyngrok

os.makedirs("models/diffusion_models", exist_ok=True)
os.makedirs("models/text_encoders", exist_ok=True)
os.makedirs("models/vae", exist_ok=True)
os.makedirs("models/loras", exist_ok=True)

print("Descargando modelos...")
!wget -q --show-progress https://huggingface.co/Comfy-Org/Krea-2/resolve/main/diffusion_models/krea2_turbo_int8_convrot.safetensors -O models/diffusion_models/krea2_turbo_int8_convrot.safetensors
!wget -q --show-progress https://huggingface.co/Comfy-Org/Krea-2/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors -O models/text_encoders/qwen3vl_4b_fp8_scaled.safetensors
!wget -q --show-progress https://huggingface.co/Comfy-Org/Krea-2/resolve/main/vae/qwen_image_vae.safetensors -O models/vae/qwen_image_vae.safetensors
!wget -q --show-progress https://huggingface.co/Comfy-Org/Krea-2/resolve/main/loras/krea2_darkbrush.safetensors -O models/loras/krea2_darkbrush.safetensors

!rm -rf /root/.cache/pip /tmp/*
!find . -type d -name "__pycache__" -exec rm -rf {} + 2>/dev/null

print("Modelos listos. Iniciando ComfyUI...")

log = open("comfyui.log", "w")
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd=COMFY, stdout=log, stderr=log
)
print("ComfyUI PID:", proc.pid)

for i in range(120):
    try:
        r = requests.get("http://localhost:8188", timeout=5)
        if r.status_code == 200:
            print("ComfyUI listo en :8188")
            break
    except:
        pass
    time.sleep(1)
else:
    print("ComfyUI no arranco. GPU activa?")

In [ ]:
print("Instalando Cloudflare Tunnel...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

cf_log = open("cloudflare.log", "w")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=cf_log, stderr=cf_log
)
time.sleep(8)

log_content = open("cloudflare.log").read()
import re
match = re.search(r'https://[a-z-]+\.trycloudflare\.com', log_content)
if match:
    url = match.group()
    print("=" * 60)
    print(f"ComfyUI en linea!")
    print(f"Enlace: {url}")
    print("=" * 60)
else:
    print("No se encontro la URL. Revisa cloudflare.log:")
    print(log_content[-1500:])

## Notas
- Cierra el tunnel en Kaggle: `File` -> `Stop execution`
- Para Kaggle: asegurate de tener Internet: ON y GPU: P100/T4

## Uso del workflow Krea-2 Turbo (precargado)
1. Abre la URL del túnel Cloudflare en tu navegador
2. Carga el workflow Krea-2 Turbo (Template > Krea-2)
3. Modelos ya descargados:
   - Diffusion: `models/diffusion_models/krea2_turbo_int8_convrot.safetensors`
   - Text encoder: `models/text_encoders/qwen3vl_4b_fp8_scaled.safetensors`
   - VAE: `models/vae/qwen_image_vae.safetensors`
   - LoRA: `models/loras/krea2_darkbrush.safetensors`
4. Dentro del subgraph:
   - `prompt_enhance`: ON por defecto (expande el prompt)
   - `enable_lora?`: true si usas darkbrush
   - `LoraLoaderModelOnly`: selecciona `krea2_darkbrush.safetensors`
   - `lora_strength`: ajusta (0.8-1.0 recomendado)
5. `CustomCombo` (canvas principal): selecciona `krea2_darkbrush` para añadir el trigger word automáticamente
6. Escribe tu prompt en `Text String (User Prompt)`
7. 8 pasos de inferencia (~137s por imagen a 2K)

## Más LoRAs (opcional)
- Descargar de https://huggingface.co/Comfy-Org/Krea-2/tree/main/loras
- Colocar en `ComfyUI/models/loras/`